# Saadi across model families and retrievers

A small matrix on a single GPU: several light instruct models from different families, each loaded through the vLLM backend, index the same synthetic document of about 1,000 tokens in 100-token chunks. For each model the notebook records whether it loads, whether its chat template and output contract work, how many entities it extracts, how many proposals are dropped, and how many model calls go to chunk extraction and how many to summary folds. Every saved state is then queried with several retrievers, and a hit table shows which retriever finds the expected chunk for each question.

**Before running:** choose *Runtime → Change runtime type → T4 GPU*. Llama and Gemma are gated on Hugging Face: accept their licenses once and store a token as the Colab secret `HF_TOKEN` (key icon on the left), or remove them from the list. Each model runs in its own subprocess so the GPU is released between families; the first run downloads about 2 GB per model.

In [ ]:
# Models: one light instruct model per family. "gated" models need HF_TOKEN; set required=False for any you may drop.
MODELS = [
    {"id": "Qwen/Qwen3-1.7B", "family": "Qwen", "required": True},
    {"id": "meta-llama/Llama-3.2-1B-Instruct", "family": "Llama", "gated": True},
    {"id": "google/gemma-3-1b-it", "family": "Gemma", "gated": True},
    {"id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "family": "SmolLM"},
    {"id": "microsoft/Phi-4-mini-instruct", "family": "Phi"},
]
# Retrievers to query every saved state with. Dense ones download their embedding model (0.4 to 2 GB).
RETRIEVERS = ["bm25", "contriever", "e5", "bge_m3"]

CHUNK_SIZE = 100
DTYPE = "half"  # a T4 has no bfloat16; use "bfloat16" on Ampere or newer
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.80
INSTALL = "saadi[vllm,dense]"

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "no HF_TOKEN: gated models will fail to download")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q {INSTALL} pandas

## The document and the questions

About 800 words, or 1,000 tokens, written so that the entity memory has something to do: people are introduced by name and later referred to by role or nickname, and places change hands between paragraphs. Each question names the chunk that answers it through a keyword, so retrieval can be scored without a judge.

In [ ]:
from pathlib import Path

DOCUMENT = """
The Halden Lake Survey began in the spring of 2021 when Ingrid Solberg, a hydrologist from Bergen, rented a boathouse on the northern shore and started taking weekly water samples. She called the project a survey, but for the first year it was one person, one canoe, and a notebook. The lake is eleven kilometres long, fed by the Vasse river in the north and drained by a narrow outlet at the village of Krog in the south. Ingrid wanted a record long enough to show whether the warm summers were changing the lake.

In the autumn of 2021 she was joined by Rafael Costa, a chemist who had worked on reservoir monitoring in Portugal. Rafael built the first laboratory in the boathouse, a bench with two spectrometers and a refrigerator for samples. He insisted on measuring phosphorus at every station, which doubled the work but gave the survey its first clear result: the Vasse delta carried far more nutrients into the lake than anyone had assumed. The director, as the villagers now called Ingrid, used that result to apply for a regional grant.

The grant paid for the Krog station, a floating platform anchored near the outlet in May 2022. It was built by a local boatwright, Henrik Aas, and a technician named Lena Vik, who had just finished her degree in Trondheim. Lena became responsible for the Krog platform and for the sensor string that hangs beneath it, recording temperature every ten minutes at six depths. When the string failed during the first winter storm, she rebuilt it with a heavier cable and a second battery, and it has not failed since.

Rafael left the survey in 2023 to return to Lisbon, and the laboratory passed to a young biologist, Priya Raman. Priya had never run a spectrometer, but she learned Rafael's methods from his notebooks and added her own: a weekly count of plankton under the microscope. Her counts showed a bloom of blue-green algae in the warm weeks of July, exactly where Rafael's phosphorus measurements had been highest. The director presented both records together at a meeting in Oslo that autumn.

The Vasse delta station was the hardest to build. The river carries sand that shifts every year, so a fixed platform was impossible. Henrik designed a buoy instead, moored to a concrete block, with the instruments inside a sealed drum. The delta buoy went into the water in April 2024, and Lena took charge of it as well, which meant a two-hour drive around the lake every second week. She calls the delta buoy the sandbox, because it has to be dug out of the river sand each spring.

Funding changed again in 2025. The regional council extended the grant on the condition that the survey published its data openly, and the director agreed. Priya wrote the data pipeline that uploads every reading to a public archive overnight. A student from Bergen, Olav Lund, joined for the summer to check the archive against the paper notebooks and found two years of temperature readings that had been logged with the wrong time zone. Correcting them moved the start of the summer stratification by six hours, which satisfied nobody but was right.

By the summer of 2026 the survey ran three stations: the boathouse, the Krog platform, and the delta buoy. Ingrid Solberg still leads it and still takes the Thursday samples by canoe. Lena Vik keeps the platform and the buoy running. Priya Raman runs the laboratory, the plankton counts, and the archive, and Olav Lund has returned as a paid assistant to extend the archive back to the first notebook of 2021. Henrik Aas no longer works for the survey, but he still tows the delta buoy ashore every November before the ice comes.

Visitors sometimes confuse the two shore stations. The boathouse is the original laboratory on the northern shore, where Priya works and where the director keeps the notebooks; the Krog platform floats at the southern outlet and has no laboratory at all, only the sensor string and a locked box of spare batteries. Lena wrote a one-page guide for new assistants that explains which station is which, how to reach each one, and whom to call when a sensor stops reporting. Olav added a map to it in 2026.

The question the survey set out to answer is not settled. The lake is warmer in August than it was in 2021, the blooms come earlier, and the phosphorus load from the Vasse has not fallen. The director's answer, when asked, is that five years is not a long record, and that the point of the survey is to still be taking samples in 2041.
""".strip()

QUESTIONS = [
    {"q": "Who founded the Halden Lake Survey?", "keyword": "Ingrid Solberg"},
    {"q": "Who built the first laboratory in the boathouse?", "keyword": "Rafael"},
    {"q": "Who is responsible for the Krog platform?", "keyword": "Lena"},
    {"q": "What does Lena call the delta buoy?", "keyword": "sandbox"},
    {"q": "Who found the time zone error in the archive?", "keyword": "Olav"},
    {"q": "Who tows the delta buoy ashore in November?", "keyword": "Henrik"},
    {"q": "Which station has no laboratory?", "keyword": "Krog platform"},
]
Path("halden.txt").write_text(DOCUMENT, encoding="utf-8")
print(len(DOCUMENT.split()), "words;", len(QUESTIONS), "questions")

## One model per subprocess

The worker loads a model through vLLM, runs the prompt-contract check, indexes the document, answers one question, and writes a JSON record. Extraction prompts equal the number of chunks; fold prompts equal the number of summaries above level 0. `model_calls` counts batched generation calls, so it is smaller than the number of prompts.

In [ ]:
from pathlib import Path

WORKER = r"""
import json, sys, time
from pathlib import Path

model_id, dtype, chunk_size, max_len, gpu_util, state_dir, out_path = sys.argv[1:8]
record = {"model": model_id, "status": "load_failed"}
t0 = time.time()
try:
    from saadi import Saadi, resolve_backend
    from saadi.prompts import check_prompt_profile

    if model_id == "mock":
        backend = resolve_backend("mock")
    else:
        backend = resolve_backend({
            "backend": "vllm", "model_id": model_id, "dtype": dtype, "max_model_len": int(max_len),
            "gpu_memory_utilization": float(gpu_util), "enforce_eager": True,
        })
    runtime = dict(getattr(backend, "runtime_metadata", {}) or {})
    record.update({"load_seconds": round(time.time() - t0, 1), "hf_commit": runtime.get("hf_commit_hash"),
                   "dtype": runtime.get("dtype"), "status": "loaded"})
    tokenizer = getattr(backend, "chat_tokenizer", None)
    record["chat_template"] = bool(tokenizer is not None and hasattr(tokenizer, "apply_chat_template"))

    text = Path("halden.txt").read_text(encoding="utf-8")
    report = check_prompt_profile("default", backend=backend, text=" ".join(text.split()[:120]))
    record["prompt_contract"] = report.ok

    t1 = time.time()
    doc = Saadi(llm=backend, state_dir=state_dir, chunk_size=int(chunk_size))
    ix = doc.index("halden.txt")
    folds = sum(len(items) for level, items in doc.history.items() if level >= 1)
    record.update({
        "index_seconds": round(time.time() - t1, 1), "chunks": ix.chunks, "entities": ix.entities,
        "rejected": ix.rejected_entities, "extraction_prompts": ix.new_chunks, "fold_prompts": folds,
        "prompts_sent": ix.prompts_sent, "model_calls": ix.model_calls,
        "entity_names": [e.name for e in list(doc.memory)[:12]],
    })
    answer = doc.answer("Who is responsible for the Krog platform?")
    record["answer"] = answer.text.strip()[:200]
    record["status"] = "ok" if (report.ok and ix.entities > 0) else "contract_failed"
    if model_id == "mock":
        record["status"] = "ok"
except Exception as exc:  # keep the matrix going
    record["error"] = f"{type(exc).__name__}: {str(exc)[:300]}"
Path(out_path).write_text(json.dumps(record, indent=2))
"""
Path("family_worker.py").write_text(WORKER.strip() + "\n", encoding="utf-8")
print("worker written")

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

Path("family_runs").mkdir(exist_ok=True)
records = []
for spec in MODELS:
    model_id = spec["id"]
    safe = model_id.replace("/", "__")
    out = Path("family_runs") / f"{safe}.json"
    state = Path("family_runs") / f"{safe}_state"
    print(f"=== {spec.get('family', '')}: {model_id}")
    proc = subprocess.run(
        [
            sys.executable,
            "family_worker.py",
            model_id,
            DTYPE,
            str(CHUNK_SIZE),
            str(MAX_MODEL_LEN),
            str(GPU_MEMORY_UTILIZATION),
            str(state),
            str(out),
        ],
        capture_output=True,
        text=True,
    )
    if out.exists():
        record = json.loads(out.read_text())
    else:
        record = {"model": model_id, "status": "crashed", "error": proc.stderr[-400:]}
    record["family"] = spec.get("family", "")
    record["required"] = bool(spec.get("required", False))
    records.append(record)
    print("   ", record["status"], "|", record.get("error", ""))

## Model table

In [ ]:
import pandas as pd

columns = [
    "family",
    "model",
    "status",
    "load_seconds",
    "chat_template",
    "prompt_contract",
    "chunks",
    "entities",
    "rejected",
    "extraction_prompts",
    "fold_prompts",
    "prompts_sent",
    "model_calls",
    "index_seconds",
    "hf_commit",
]
table = pd.DataFrame(records).reindex(columns=columns)
display(table)  # noqa: F821
for record in records:
    if record.get("entity_names"):
        print(f"{record['model']}: {', '.join(record['entity_names'])}")
        print("   answer:", record.get("answer", ""))

## Retriever matrix

Each saved state is loaded without a model and queried with every retriever. A hit means the expected keyword appears in the top-3 chunks. The memory and summary channels differ per model because the states differ; the text channel is the same document for all.

In [ ]:
from saadi import Saadi

rows = []
for record in records:
    if record.get("status") not in ("ok", "contract_failed"):
        continue
    state = Path("family_runs") / f"{record['model'].replace('/', '__')}_state"
    for retriever in RETRIEVERS:
        try:
            doc = Saadi.load(state, retriever=retriever)
            hits = 0
            for item in QUESTIONS:
                top = doc.retrieve(item["q"], k=3)
                hits += any(item["keyword"] in hit.text for hit in top)
            rows.append({"model": record["model"], "retriever": retriever, "hits@3": f"{hits}/{len(QUESTIONS)}"})
        except Exception as exc:
            rows.append({"model": record["model"], "retriever": retriever, "hits@3": f"error: {type(exc).__name__}"})
matrix = pd.DataFrame(rows).pivot(index="model", columns="retriever", values="hits@3")
display(matrix)  # noqa: F821

## Verdict

In [ ]:
failed_required = [r["model"] for r in records if r["required"] and r["status"] != "ok"]
skipped = [r["model"] for r in records if r["status"] in ("load_failed", "crashed")]
print("loaded and passed:", [r["model"] for r in records if r["status"] == "ok"])
print("loaded, contract or extraction failed:", [r["model"] for r in records if r["status"] == "contract_failed"])
print("did not load:", skipped)
assert not failed_required, f"required models failed: {failed_required}"
print("Required models pass.")

## Reading the results

- `status = contract_failed` means the model loaded and ran but either did not follow the output contract or extracted no entities. That is a model limitation at this size, not a packaging problem; a larger model of the same family usually passes.
- `rejected` counts proposals whose anchor text was not verbatim in the chunk. Small models produce more of them.
- `fold_prompts` is the summarization cost: with 100-token chunks and the default folding factor of 5, about one fold per five chunks plus one per five level-1 summaries.
- A `did not load` entry for a gated model means the Hugging Face token is missing or the license has not been accepted.
- To check a specific model the paper used, set `MODELS` to one entry with its id and `DTYPE = "bfloat16"` on a suitable GPU.